# Finetuning

Partial fine-tune of InternVideo2-1B-s2 on the TVPReid train split, then the same test metrics as the zero-shot notebook. Clones the shawaf repo and OpenGVLab/InternVideo. Trains the projection, the last 4 video blocks, and the last 2 BERT layers with video-text contrastive loss only. Batch 1, gradient accumulation 8, fp16, DeepSpeed off. The 1B-s2 checkpoint is gated: accept the license on Hugging Face, then attach a Kaggle secret named `hugging_face`. The install cell loads that secret.


In [ ]:
from pathlib import Path

SUBSETS = ["prid", "ilids", "duke"]
NUM_FRAMES = 4
EPOCHS = 5
LEARNING_RATE = 5e-6
BATCH = 1
GRAD_ACCUM = 8
VISION_LAST_BLOCKS = 4
TEXT_LAST_LAYERS = 2
DEVICE = "cuda"
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("results/finetune")
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = WORK / "internvideo2_s2_tvpreid"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(
    "Finetune InternVideo2-1B-s2",
    f"epochs={EPOCHS} lr={LEARNING_RATE} batch={BATCH} accum={GRAD_ACCUM}",
    flush=True,
)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

SHAWAF_URL = "https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-.git"
INTERNVIDEO_URL = "https://github.com/OpenGVLab/InternVideo.git"
EXPECTED = "0.1.16"


def run(cmd, cwd=None):
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)


ON_KAGGLE = Path("/kaggle/working").is_dir()
if ON_KAGGLE:
    shawaf_dir = Path("/kaggle/working/shawaf-vlm")
    intern_dir = Path("/kaggle/working/InternVideo")
    if shawaf_dir.exists():
        run(["git", "fetch", "origin"], cwd=shawaf_dir)
        run(["git", "reset", "--hard", "origin/main"], cwd=shawaf_dir)
    else:
        run(["git", "clone", SHAWAF_URL, str(shawaf_dir)])
    if intern_dir.exists():
        run(["git", "fetch", "origin"], cwd=intern_dir)
        run(["git", "reset", "--hard", "origin/main"], cwd=intern_dir)
    else:
        run(["git", "clone", INTERNVIDEO_URL, str(intern_dir)])
    run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{shawaf_dir}[all]"])
    run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "einops", "timm", "av", "imageio", "librosa", "soundfile",
            "pandas", "pyyaml", "scipy", "wandb",
        ]
    )
    try:
        run([sys.executable, "-m", "pip", "install", "-q", "decord"])
    except subprocess.CalledProcessError:
        print("decord wheel failed; the fine-tune path stubs it and uses PyAV", flush=True)
else:
    here = Path.cwd().resolve()
    shawaf_dir = here
    for candidate in [here, *here.parents]:
        if (candidate / "shawaf_vlm").is_dir() and (candidate / "pyproject.toml").is_file():
            shawaf_dir = candidate
            break
    intern_dir = shawaf_dir / "InternVideo"
    if not intern_dir.is_dir():
        run(["git", "clone", INTERNVIDEO_URL, str(intern_dir)], cwd=shawaf_dir)

os.environ["INTERNVIDEO_ROOT"] = str(intern_dir)
if str(shawaf_dir) not in sys.path:
    sys.path.insert(0, str(shawaf_dir))

for name in list(sys.modules):
    if name == "shawaf_vlm" or name.startswith("shawaf_vlm."):
        del sys.modules[name]

import shawaf_vlm

print("shawaf_vlm", shawaf_vlm.__version__, shawaf_vlm.__file__, flush=True)
print("INTERNVIDEO_ROOT", os.environ["INTERNVIDEO_ROOT"], flush=True)
if shawaf_vlm.__version__ != EXPECTED:
    raise RuntimeError(
        f"Expected shawaf_vlm {EXPECTED}, found {shawaf_vlm.__version__}. "
        "Restart the session and run this cell again after origin/main updates."
    )

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not token and Path("/kaggle/working").is_dir():
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("hugging_face")
if not token:
    raise RuntimeError(
        "Hugging Face token missing. On Kaggle, add a secret named hugging_face "
        "and attach it to this notebook."
    )
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token
from huggingface_hub import login

login(token=token, add_to_git_credential=False)
print("Hugging Face token loaded", flush=True)


In [ ]:
from shawaf_vlm.data.tvpreid import download_tvpreid, load_tvpreid

DATA_ROOT = None
for split in ("train", "val"):
    DATA_ROOT = download_tvpreid(configs=tuple(SUBSETS), split=split)
print("TVPReid root", DATA_ROOT, flush=True)


In [ ]:
from shawaf_vlm.finetune_internvideo2 import (
    launch_training,
    write_retrieval_json,
    write_s2_config,
)
from shawaf_vlm.models.internvideo2_s2 import download_s2_checkpoint

train_splits = [load_tvpreid(subset, split="train", root=DATA_ROOT) for subset in SUBSETS]
val_splits = [load_tvpreid(subset, split="val", root=DATA_ROOT) for subset in SUBSETS]
train_json = write_retrieval_json(train_splits, WORK / "tvpreid_train.json")
val_json = write_retrieval_json(val_splits, WORK / "tvpreid_val.json")
weights = download_s2_checkpoint()
config_path = write_s2_config(
    WORK / "tvpr_s2_config.py",
    train_json=train_json,
    val_json=val_json,
    pretrained_path=weights,
    output_dir=OUTPUT_DIR,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH,
    num_frames=NUM_FRAMES,
)
launch_training(
    config_path,
    vision_last_blocks=VISION_LAST_BLOCKS,
    text_last_layers=TEXT_LAST_LAYERS,
    grad_accum=GRAD_ACCUM,
)
print("Checkpoint", OUTPUT_DIR / "ckpt_latest.pth", flush=True)


In [ ]:
import json

from shawaf_vlm.eval_loop import evaluate_text_to_tracklet
from shawaf_vlm.metrics import format_metrics
from shawaf_vlm.models.internvideo2_s2 import InternVideo2S2Encoder
from shawaf_vlm.models.runtime import ensure_cuda_healthy

checkpoint = OUTPUT_DIR / "ckpt_latest.pth"
if not checkpoint.is_file():
    raise FileNotFoundError(f"Training did not write {checkpoint}")
ensure_cuda_healthy(DEVICE)
encoder = InternVideo2S2Encoder(device=DEVICE, checkpoint=checkpoint, name="internvideo2_s2_1b_ft")
rows = []
for subset in SUBSETS:
    splits = load_tvpreid(subset, split="test", root=DATA_ROOT)
    metrics = evaluate_text_to_tracklet(
        encoder,
        splits,
        num_frames=NUM_FRAMES,
        batch_size=1,
        text_batch_size=8,
        junk_same_camera=False,
        frame_cache=DATA_ROOT / "frame_cache",
        frame_sample="middle",
    )
    print(subset, flush=True)
    print(format_metrics(metrics), flush=True)
    payload = {"model": "internvideo2_s2_1b_ft", "subset": subset, "split": "test", "metrics": metrics}
    out = WORK / f"finetune_{subset}_test.json"
    out.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    rows.append(payload)
    print("Wrote", out, flush=True)


In [ ]:
print(f"{'subset':8} {'R@1':7} {'R@5':7} {'R@10':7} {'mAP':7} {'MdR':7} {'video_s':8} {'peak_gb':7}")
for row in rows:
    metrics = row["metrics"]
    print(
        f"{row['subset']:8} "
        f"{metrics['Rank-1']:7.2f} {metrics.get('Rank-5', 0):7.2f} "
        f"{metrics.get('Rank-10', 0):7.2f} {metrics['mAP']:7.2f} "
        f"{metrics['MdR']:7.1f} {metrics['video_s']:8.1f} {metrics['peak_gpu_gb']:7.2f}"
    )
